In [34]:
import duckdb
from dotenv import load_dotenv
import os
import s3fs
import polars as pl
import boto3
load_dotenv()

pl.Config.set_fmt_str_lengths(50)

polars.config.Config

In [26]:

s3 = s3fs.S3FileSystem(
    key=os.getenv("S3_ACCESS_KEY"),
    secret=os.getenv("S3_SECRET_KEY"),
    token=os.getenv("AWS_SESSION_TOKEN"),  # optional but important if using temp creds
    client_kwargs={
        "region_name": "ca-central-1"
    }
)



In [27]:
# s3 setup using boto3
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("S3_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("S3_SECRET_KEY"),
    aws_session_token=os.getenv("AWS_SESSION_TOKEN"),  # optional but important if using temp creds
    region_name='ca-central-1'
)

In [28]:
# scan s3 bucket using duckdb

con = duckdb.connect()

bucket = os.getenv("S3_BUCKET")

snp500 = con.execute(f"""
SELECT *
FROM read_csv_auto('s3://{bucket}/post-earnings-forecast/**/*.csv')

""").df()
snp500

,#,Company,Symbol,Weight,Price,Chg,% Chg
0,1,NVIDIA Corp,NVDA,7.69%,215.33,-4.18,-1.90%
1,2,Apple Inc,AAPL,6.69%,308.82,3.83,1.26%
2,3,Microsoft Corp,MSFT,4.58%,418.57,-0.52,-0.12%
3,4,Amazon.com Inc,AMZN,4.22%,266.32,-2.14,-0.80%
4,5,Alphabet Inc,GOOGL,3.53%,382.97,-4.69,-1.21%
...,...,...,...,...,...,...,...
498,499,Pool Corp,POOL,0.01%,184.64,2.95,1.62%
499,500,Conagra Brands Inc,CAG,0.01%,13.56,0.18,1.35%
500,501,Campbell's Company/The,CPB,0.01%,20.58,0.53,2.64%
501,502,News Corp,NWS,0.01%,29.68,-0.40,-1.33%


# Earnings 


In [29]:
#  delta table for earnings
DELTA_EARNINGS = f"s3://{os.getenv('S3_BUCKET')}/post-earnings-forecast/earnings_delta/"


# storage options for delta lake
storage_options = {
    "AWS_ACCESS_KEY_ID": os.getenv("AWS_ACCESS_KEY_ID"),
    "AWS_SECRET_ACCESS_KEY": os.getenv("AWS_SECRET_ACCESS_KEY"),
    "AWS_REGION": "ca-central-1",
}


delta_earnings_df = pl.scan_delta(DELTA_EARNINGS, storage_options=storage_options).collect()
delta_earnings_df


delta_earnings_df




fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from
date,date,str,str,str,str,str,str,i8,i32,str,str
2015-09-30,2015-11-06,"""1.85""","""1.85""","""0""","""0""","""post-market""","""BRK-B""",9,2015,"""2015Q3""","""20151106T0000"""
2019-03-31,2019-05-04,"""2.26""","""2.29""","""-0.03""","""-1.31""","""pre-market""","""BRK-B""",3,2019,"""2019Q1""","""20190504T0000"""
2017-09-30,2017-11-03,"""1.4""","""1.58""","""-0.18""","""-11.3924""","""post-market""","""BRK-B""",9,2017,"""2017Q3""","""20171103T0000"""
2016-12-31,2017-02-25,"""1.78""","""1.83""","""-0.05""","""-2.7322""","""pre-market""","""BRK-B""",12,2016,"""2016Q4""","""20170225T0000"""
2023-06-30,2023-08-05,"""4.62""","""3.87""","""0.75""","""19.3798""","""pre-market""","""BRK-B""",6,2023,"""2023Q2""","""20230805T0000"""
…,…,…,…,…,…,…,…,…,…,…,…
2014-09-30,2014-10-22,"""0.02""","""0.02""","""0""","""0""","""post-market""","""FTNT""",9,2014,"""2014Q3""","""20141022T0000"""
2025-09-30,2025-11-05,"""0.74""","""0.63""","""0.11""","""17.4603""","""post-market""","""FTNT""",9,2025,"""2025Q3""","""20251105T0000"""
2025-03-31,2025-05-07,"""0.58""","""0.53""","""0.05""","""9.434""","""post-market""","""FTNT""",3,2025,"""2025Q1""","""20250507T0000"""


In [24]:
delta_earnings_df.filter(
    pl.col('symbol') == 'AAPL'
)

fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from
date,date,str,str,str,str,str,str,i8,i32,str,str
2020-09-30,2020-10-29,"""0.73""","""0.7""","""0.03""","""4.2857""","""post-market""","""AAPL""",9,2020,"""2020Q3""","""20201029T0000"""
2019-03-31,2019-04-30,"""0.62""","""0.59""","""0.03""","""5.0847""","""post-market""","""AAPL""",3,2019,"""2019Q1""","""20190430T0000"""
2017-09-30,2017-11-02,"""0.5175""","""0.4675""","""0.05""","""10.6952""","""post-market""","""AAPL""",9,2017,"""2017Q3""","""20171102T0000"""
2016-12-31,2017-01-31,"""0.84""","""0.8025""","""0.0375""","""4.6729""","""post-market""","""AAPL""",12,2016,"""2016Q4""","""20170131T0000"""
2015-09-30,2015-10-27,"""0.49""","""0.47""","""0.02""","""4.2553""","""post-market""","""AAPL""",9,2015,"""2015Q3""","""20151027T0000"""
…,…,…,…,…,…,…,…,…,…,…,…
2017-06-30,2017-08-01,"""0.4175""","""0.3925""","""0.025""","""6.3694""","""post-market""","""AAPL""",6,2017,"""2017Q2""","""20170801T0000"""
2016-09-30,2016-10-25,"""0.4175""","""0.415""","""0.0025""","""0.6024""","""post-market""","""AAPL""",9,2016,"""2016Q3""","""20161025T0000"""
2016-03-31,2016-04-26,"""0.475""","""0.5""","""-0.025""","""-5""","""post-market""","""AAPL""",3,2016,"""2016Q1""","""20160426T0000"""


In [30]:
# total syms in earnings delta
delta_earnings_df.select(pl.col('symbol').n_unique())

symbol
u32
503


# Transcripts 

In [41]:
DELTA_TRANSCRIPTS = f"s3://{os.getenv('S3_BUCKET')}/post-earnings-forecast/transcripts_delta/"
# polars options 

df_tx = pl.scan_delta(DELTA_TRANSCRIPTS, storage_options=storage_options).collect()
df_tx.filter(pl.col('symbol') == 'NVDA').to_pandas().head()

,fiscalDateEnding,reportedDate,reportedEPS,estimatedEPS,surprise,surprisePercentage,reportTime,symbol,fiscal_month,fiscal_year,av_quarter,time_from,transcript
0,2014-10-31,2014-11-06,0.008,0.007,0.001,14.2857,post-market,AMP,6,2015,2015Q2,20150722T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
1,2016-07-31,2016-08-11,0.01,0.009,0.001,11.1111,post-market,NVDA,7,2016,2016Q3,20160811T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
2,2018-07-31,2018-08-16,0.049,0.044,0.005,11.3636,post-market,AMP,9,2018,2018Q3,20181023T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
3,2018-10-31,2018-11-15,0.046,0.047,-0.001,-2.1277,post-market,AMP,12,2018,2018Q4,20190130T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."
4,2020-07-31,2020-08-19,0.055,0.049,0.006,12.2449,post-market,AMP,9,2020,2020Q3,20201028T0000,"[{'speaker': 'Operator', 'title': 'Operator', ..."


In [42]:
df_tx['symbol'].n_unique()

169

# OHLVC


In [45]:
DELTA_OHLVC = f"s3://{bucket}/post-earnings-forecast/ohlcv_delta/"
df = pl.read_delta(DELTA_OHLVC, storage_options=storage_options)
df.head()

symbol,date,open,high,low,close,volume
str,date,f64,f64,f64,f64,i64
"""BRK-B""",2011-01-27,82.739998,83.239998,82.650002,83.029999,2817800
"""BRK-B""",2013-02-28,101.440002,102.330002,101.309998,102.160004,8875500
"""BRK-B""",2013-03-01,101.449997,102.25,100.440002,102.050003,4812800
"""BRK-B""",2013-03-06,103.290001,103.559998,102.760002,103.239998,3316300
"""BRK-B""",2013-03-15,103.669998,103.790001,102.790001,102.790001,7253800


In [46]:
df['symbol'].n_unique()

503

In [47]:
df.shape

(4385666, 7)

In [63]:
# filter frame from 2010
df.filter(pl.col('date') >=  pl.lit("2010-01-01").str.to_date()).shape

(1954181, 7)